# KavachX AI/ML Technical Assessment — Task 2: Model Training
### Fine-tune YOLOv8n on Factory PPE Dataset (Helmet vs No_Helmet)
This notebook trains a lightweight YOLOv8n detector at 416px for ~15 epochs on Google Colab GPU with CCTV-tailored data augmentations.

## 1. Environment Setup & GPU Check

In [ ]:
!nvidia-smi
!pip install ultralytics==8.3.40 opencv-python-headless pillow

## 2. Dataset Preparation
Upload the `data/ppe` directory or extract the zip archive containing 240 train / 60 val frames.

In [ ]:
import os

# Create data.yaml
yaml_content = """
path: /content/data/ppe
train: train/images
val: val/images

nc: 2
names:
  0: helmet
  1: no_helmet
"""
os.makedirs("/content/data/ppe", exist_ok=True)
with open("/content/data/ppe/data.yaml", "w") as f:
    f.write(yaml_content.strip())

print("data.yaml configured successfully.")

## 3. Data Augmentation Strategy & Training

### CCTV Augmentation Rationale:
- `mosaic=1.0`: Combines 4 images into one, reproducing crowded industrial shop floors with multiple workers.
- `mixup=0.1`: Linear blending of images to make the model robust against mutual worker occlusions.
- `hsv_h=0.015, hsv_s=0.5, hsv_v=0.3`: Adapts to lighting shifts (harsh sodium lights, daylight, shadows) in factory settings.
- `degrees=5.0`: Handles minor camera tilt/shake on mounting poles.
- `translate=0.1`: Simulates workers positioned at various parts of the field of view.
- `scale=0.3`: Captures scale variation as workers move nearer or farther from fixed CCTV lenses.
- `perspective=0.0005`: Corrects for steep top-down angled distortion characteristic of overhead security cameras.
- `flipud=0.0`: STRICTLY DISABLED — workers on a factory floor never appear inverted.
- `fliplr=0.5`: Horizontal flipping reflects valid movement directions across the camera FOV.

In [ ]:
from ultralytics import YOLO

# Load baseline YOLOv8n COCO weights
model = YOLO('yolov8n.pt')

# Train for 15 epochs at 416px
results = model.train(
    data='/content/data/ppe/data.yaml',
    imgsz=416,
    epochs=15,
    batch=16,
    device=0,
    name='ppe_yolov8n',
    # CCTV-specific augmentations
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    perspective=0.0005,
    flipud=0.0,
    fliplr=0.5,
    plots=True
)

## 4. Evaluation: Baseline mAP@50 and Recall

In [ ]:
# Validate on validation set
metrics = model.val(data='/content/data/ppe/data.yaml', imgsz=416)

print(f"\n=== Validation Metrics ===")
print(f"Overall mAP@50:     {metrics.box.map50:.4f}")
print(f"Overall mAP@50-95:  {metrics.box.map:.4f}")
print(f"Precision:          {metrics.box.mp:.4f}")
print(f"Recall:             {metrics.box.mr:.4f}")

# Class-specific recall
class_names = ['helmet', 'no_helmet']
for idx, name in enumerate(class_names):
    if idx < len(metrics.box.r):
        print(f"Class '{name}' Recall: {metrics.box.r[idx]:.4f}")

## 5. Export Model to ONNX

In [ ]:
# Export trained weights to ONNX format
best_model = YOLO('runs/detect/ppe_yolov8n/weights/best.pt')
onnx_path = best_model.export(format='onnx', imgsz=416, dynamic=False)
print(f"Exported ONNX model to: {onnx_path}")